# Demo kiểm tra bộ nhớ hội thoại Redis

Notebook này ưu tiên kiểm tra logic memory nhẹ:
- Không tải LLM.
- Không tải embedding.
- Có thể đọc Redis nếu Redis đang chạy, nhưng không bắt buộc.

Khi muốn chạy chatbot thật, dùng command ở cell cuối.

In [1]:
import sys
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from chat_rag_redis import inspect_memory_usage, format_history

In [2]:
history = [
    {"role": "user", "content": "Cây sầu riêng sau mưa bị vàng lá, rễ yếu thì dùng gì?"},
    {"role": "assistant", "content": "Có thể dùng PHOSPHOROUS ACID 400SL theo đúng hướng dẫn trên nhãn."},
]
question = "Sản phẩm đó dùng như thế nào?"

print(format_history(history))
debug = inspect_memory_usage(question, history)
debug

Người dùng: Cây sầu riêng sau mưa bị vàng lá, rễ yếu thì dùng gì?
Trợ lý: Có thể dùng PHOSPHOROUS ACID 400SL theo đúng hướng dẫn trên nhãn.


{'decision': 'likely_used_memory',
 'explanation': 'Câu hỏi có dấu hiệu hỏi nối tiếp và Redis có lịch sử hội thoại.',
 'has_history_before_answer': True,
 'history_message_count': 2,
 'history_user_turn_count': 1,
 'history_assistant_turn_count': 1,
 'matched_follow_up_markers': ['đó',
  'sản phẩm đó',
  'như thế nào',
  'dùng như thế nào'],
 'answer_overlap_with_previous_assistant': False}

In [3]:
try:
    import redis
    client = redis.Redis.from_url("redis://localhost:6379/0", decode_responses=True)
    pong = client.ping()
    print("Redis ping:", pong)
    print("Ví dụ các key rag_chat:*:")
    print(client.keys("rag_chat:*"))
except Exception as exc:
    print("Chưa kết nối Redis được, bỏ qua phần đọc Redis.")
    print(type(exc).__name__, exc)

Redis ping: True
Ví dụ các key rag_chat:*:
[]


In [4]:
# Command chạy thật trên máy/Kaggle khi Redis đã sẵn sàng.
# Giữ max-new-tokens thấp để test nhẹ.
print('''python3 scripts/chat_rag_redis.py \
  --redis-url redis://localhost:6379/0 \
  --session-id demo-memory-1 \
  --embedding bge-m3 \
  --generator hf \
  --llm qwen3_4b \
  --device cuda \
  --llm-device cuda \
  --top-k 3 \
  --max-new-tokens 256 \
  --show-memory-debug \
  --question "Sản phẩm đó dùng như thế nào?"''')

python3 scripts/chat_rag_redis.py   --redis-url redis://localhost:6379/0   --session-id demo-memory-1   --embedding bge-m3   --generator hf   --llm qwen3_4b   --device cuda   --llm-device cuda   --top-k 3   --max-new-tokens 256   --show-memory-debug   --question "Sản phẩm đó dùng như thế nào?"
